# Context Engineering 完整演示

本 notebook 用纯 Python 演示上下文工程的核心流程：上下文块、窗口管理、记忆、RAG、注入策略、压缩与评估。


In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from collections import Counter
import math
import re

## 1. 上下文块与优先级

把输入拆成可管理的 ContextBlock，后续才能做裁剪、排序、注入和评估。


In [2]:
@dataclass
class ContextBlock:
    name: str
    content: str
    priority: int = 1
    source: str = "manual"

    @property
    def tokens(self) -> int:
        chinese = sum(1 for c in self.content if "\u4e00" <= c <= "\u9fff")
        other = len(self.content) - chinese
        return int(chinese * 1.5 + other * 0.25)

blocks = [
    ContextBlock("system", "你是课程问答助手，只基于给定资料回答。", priority=5),
    ContextBlock("profile", "用户是软件学院学生，正在学习 RAG。", priority=3),
    ContextBlock("doc", "RAG 包括文档切分、向量检索、上下文注入和答案生成。", priority=4),
    ContextBlock("history", "上一轮用户问过上下文窗口限制。", priority=2),
]

for block in blocks:
    print(block.name, block.priority, block.tokens, block.content)


system 5 26 你是课程问答助手，只基于给定资料回答。
profile 3 21 用户是软件学院学生，正在学习 RAG。
doc 4 31 RAG 包括文档切分、向量检索、上下文注入和答案生成。
history 2 21 上一轮用户问过上下文窗口限制。


## 2. Token 预算与窗口管理


In [3]:
def select_context(blocks: List[ContextBlock], budget: int) -> List[ContextBlock]:
    selected = []
    used = 0
    for block in sorted(blocks, key=lambda x: x.priority, reverse=True):
        if used + block.tokens <= budget:
            selected.append(block)
            used += block.tokens
    return selected

selected = select_context(blocks, budget=70)
print("选中的上下文：")
for block in selected:
    print("-", block.name, block.tokens)


选中的上下文：
- system 26
- doc 31


## 3. 短期记忆与长期记忆


In [4]:
class MemoryStore:
    def __init__(self):
        self.short_term = []
        self.long_term = []

    def add_turn(self, role, content):
        self.short_term.append({"role": role, "content": content})
        self.short_term = self.short_term[-6:]

    def remember(self, fact, tags=None):
        self.long_term.append({"fact": fact, "tags": tags or []})

    def recall(self, query):
        return [item for item in self.long_term if any(tag in query for tag in item["tags"])]

memory = MemoryStore()
memory.add_turn("user", "RAG 的检索结果应该放在哪里？")
memory.add_turn("assistant", "通常放在用户问题前或专门的资料区。")
memory.remember("用户正在准备上下文工程课程展示", tags=["课程", "展示"])
print(memory.short_term)
print(memory.recall("课程案例"))


[{'role': 'user', 'content': 'RAG 的检索结果应该放在哪里？'}, {'role': 'assistant', 'content': '通常放在用户问题前或专门的资料区。'}]
[{'fact': '用户正在准备上下文工程课程展示', 'tags': ['课程', '展示']}]


## 4. 简化 RAG：切分、检索、生成上下文


In [5]:
docs = {
    "rag": "RAG 通过检索外部知识增强模型回答，常见流程包括切分、索引、召回、重排和生成。",
    "memory": "记忆系统分为短期记忆、长期记忆和情景记忆，用于保持多轮任务连续性。",
    "compression": "上下文压缩用于在 token 预算有限时保留关键事实，删除冗余描述。",
}

def tokenize(text):
    return set(re.findall(r"[a-zA-Z]+|[\u4e00-\u9fff]{2,}", text.lower()))

def retrieve(query, top_k=2):
    q = tokenize(query)
    scored = []
    for name, text in docs.items():
        score = len(q & tokenize(text)) / max(len(q), 1)
        scored.append((score, name, text))
    return sorted(scored, reverse=True)[:top_k]

query = "RAG 如何管理检索上下文？"
retrieved = retrieve(query)
for score, name, text in retrieved:
    print(f"{name}: {score:.2f} {text}")


rag: 0.50 RAG 通过检索外部知识增强模型回答，常见流程包括切分、索引、召回、重排和生成。
memory: 0.00 记忆系统分为短期记忆、长期记忆和情景记忆，用于保持多轮任务连续性。


## 5. 上下文注入模板


In [6]:
def build_prompt(question, retrieved_docs, memory_items):
    knowledge = "\n".join(f"[{name}] {text}" for _, name, text in retrieved_docs)
    memories = "\n".join(item["fact"] for item in memory_items) or "无"
    return f"""你是上下文工程课程助手。

长期记忆：
{memories}

检索资料：
{knowledge}

用户问题：{question}

请基于资料回答，缺失信息要说明。"""

prompt = build_prompt(query, retrieved, memory.recall("课程展示"))
print(prompt)


你是上下文工程课程助手。

长期记忆：
用户正在准备上下文工程课程展示

检索资料：
[rag] RAG 通过检索外部知识增强模型回答，常见流程包括切分、索引、召回、重排和生成。
[memory] 记忆系统分为短期记忆、长期记忆和情景记忆，用于保持多轮任务连续性。

用户问题：RAG 如何管理检索上下文？

请基于资料回答，缺失信息要说明。


## 6. 压缩与质量评估


In [7]:
def compress_context(text, max_sentences=2):
    sentences = re.split(r"[。！？]", text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return "。".join(sentences[:max_sentences]) + "。"

def evaluate_context(prompt, required_terms):
    return {term: (term in prompt) for term in required_terms}

compressed = compress_context(prompt, max_sentences=3)
print(compressed)
print(evaluate_context(prompt, ["RAG", "检索", "长期记忆", "用户问题"]))


你是上下文工程课程助手。长期记忆：
用户正在准备上下文工程课程展示

检索资料：
[rag] RAG 通过检索外部知识增强模型回答，常见流程包括切分、索引、召回、重排和生成。[memory] 记忆系统分为短期记忆、长期记忆和情景记忆，用于保持多轮任务连续性。
{'RAG': True, '检索': True, '长期记忆': True, '用户问题': True}


## 7. 教学案例建议

- 课程问答机器人：用 RAG 注入教材片段。
- 论文阅读助手：按章节检索、压缩和引用。
- 编程助教：把错误日志、代码片段、历史尝试组织进上下文。
- 多轮项目助手：短期记忆保存当前任务，长期记忆保存项目约束。


## 7. 上下文类型分层：Input / Runtime / Compression / Isolation / Long-term

较新的上下文工程资料会把上下文分成多类：启动时注入的输入上下文、每次运行传入的运行时上下文、接近窗口上限时的压缩上下文、通过子任务隔离出的局部上下文，以及跨会话保留的长期记忆。这个分类比“把所有资料拼进 prompt”更适合长任务。


In [ ]:
@dataclass
class ContextLayer:
    name: str
    scope: str
    examples: List[str]
    risk: str

layers = [
    ContextLayer("input_context", "每次启动固定加载", ["system prompt", "课程规则", "项目约定"], "过长会挤占用户任务空间"),
    ContextLayer("runtime_context", "单次运行动态传入", ["用户身份", "权限", "当前文件"], "权限上下文必须可信"),
    ContextLayer("compressed_context", "窗口接近上限时生成", ["对话摘要", "检索摘要"], "压缩可能丢失关键证据"),
    ContextLayer("isolated_context", "子任务/子代理内部", ["并行研究", "大文件分析"], "需要只返回结论和证据"),
    ContextLayer("long_term_memory", "跨会话持久化", ["用户偏好", "项目事实", "历史决策"], "写入频率和更新策略要评估"),
]

for layer in layers:
    print(f"{layer.name}: {layer.scope} | 示例={layer.examples} | 风险={layer.risk}")


## 8. RAG 检索评估：Hit Rate 与 MRR

只看最终回答不够。RAG 应先评估“检索有没有把正确证据找出来”。下面用一个小型标注集演示 Hit Rate 和 MRR。


In [ ]:
eval_set = [
    {"query": "RAG 包含哪些流程？", "gold": "rag"},
    {"query": "长期记忆和短期记忆有什么区别？", "gold": "memory"},
    {"query": "如何减少上下文 token？", "gold": "compression"},
]

def ranked_retrieve(query, top_k=3):
    q = tokenize(query)
    scored = []
    for name, text in docs.items():
        score = len(q & tokenize(text)) / max(len(q), 1)
        scored.append((score, name, text))
    return sorted(scored, reverse=True)[:top_k]

def evaluate_retriever(dataset, top_k=2):
    hits = []
    reciprocal_ranks = []
    for item in dataset:
        ranked = ranked_retrieve(item["query"], top_k=top_k)
        names = [name for _, name, _ in ranked]
        hit = item["gold"] in names
        hits.append(hit)
        if hit:
            reciprocal_ranks.append(1 / (names.index(item["gold"]) + 1))
        else:
            reciprocal_ranks.append(0)
        print(item["query"], "=>", names, "gold=", item["gold"], "hit=", hit)
    return {"hit_rate": sum(hits) / len(hits), "mrr": sum(reciprocal_ranks) / len(reciprocal_ranks)}

print(evaluate_retriever(eval_set, top_k=2))


## 9. 引用、新鲜度与冲突检查

生产 RAG 不应只返回答案，还要返回来源、更新时间和冲突提示。这样学生能理解：上下文工程的目标不是“塞更多资料”，而是让模型看到可信、可追踪、不过期的信息。


In [ ]:
@dataclass
class SourceChunk:
    source: str
    content: str
    updated_at: str

source_chunks = [
    SourceChunk("policy_2026", "毕业设计提交截止日期为 2026-05-30。", "2026-03-01"),
    SourceChunk("policy_2024", "毕业设计提交截止日期为 2024-05-20。", "2024-03-01"),
]

def build_cited_context(chunks: List[SourceChunk]) -> str:
    lines = []
    for idx, chunk in enumerate(chunks, 1):
        lines.append(f"[{idx}] source={chunk.source} updated={chunk.updated_at}\n{chunk.content}")
    return "\n\n".join(lines)

def detect_conflict(chunks: List[SourceChunk]) -> bool:
    dates = set(re.findall(r"20\d{2}-\d{2}-\d{2}", " ".join(c.content for c in chunks)))
    return len(dates) > 1

print(build_cited_context(source_chunks))
print("存在冲突：", detect_conflict(source_chunks))
print("建议：优先使用 updated_at 最新的政策，并在回答中标注来源。")
